# Official Naive Baselines

Ovaj notebook je sluzbeni execution path za naive baselinee u radu.
Cilj mu je pokazati cijeli tok: od metadata i hourly podataka do predikcija, metrika i spremanja artefakata.

## Sto se ovdje dogada

U ovom notebooku radimo cetiri stvari:

1. Ucitamo `experiment_metadata.json` i potvrdimo da radimo sluzbeni forecasting task `24h -> 1h ahead` nad targetom `volume`.
2. Iz `station_hourly` CSV datoteka ponovno izgradimo isti sample universe koji koristi glavni experiment pipeline.
3. Izracunamo dvije naive predikcije:
   - `naive_last_value`
   - `naive_seasonal_24h`
4. Izracunamo metrike i spremimo iste artefakte koje koriste i ostali eksperimenti.

## Configuration

Promijeni vrijednosti u sljedecoj celiji ako zelis koristiti drugi output folder ili drugo ime runa.

In [1]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "diplomski").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "diplomski").exists():
    raise RuntimeError("Run this notebook from the project root or from notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from diplomski.evaluation import (
    SUPPORTED_FEATURE_COUNT,
    build_prediction_frame,
    build_results_payload,
    compute_regression_metrics,
    configure_run_logger,
    load_experiment_metadata,
    make_run_name,
    save_predictions_csv,
    save_results_payload,
    validate_official_task,
    write_empty_communication_rounds,
)
from diplomski.preprocessing.config import ExperimentConfig
from diplomski.preprocessing.experiment_utils import (
    SplitDataFrame,
    build_station_splits_from_hourly,
    build_windows_by_station,
    clean_split_for_windows,
    discover_hourly_files,
)

RUN_NAME = os.environ.get("DIPLOMSKI_RUN_NAME", f"notebook_naive_{make_run_name()}")
HOURLY_DIR = Path(
    os.environ.get(
        "DIPLOMSKI_HOURLY_DIR",
        str(PROJECT_ROOT / "data" / "processed" / "station_hourly"),
    )
).resolve()
METADATA_PATH = Path(
    os.environ.get(
        "DIPLOMSKI_EXPERIMENT_METADATA",
        str(PROJECT_ROOT / "data" / "processed" / "experiments" / "artifacts" / "experiment_metadata.json"),
    )
).resolve()
OUTPUT_ROOT = Path(
    os.environ.get(
        "DIPLOMSKI_OUTPUT_ROOT",
        str(PROJECT_ROOT / "data" / "processed" / "results"),
    )
).resolve()
GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()
BASELINE_METHODS = ("naive_last_value", "naive_seasonal_24h")

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUN_NAME={RUN_NAME}")
print(f"HOURLY_DIR={HOURLY_DIR}")
print(f"METADATA_PATH={METADATA_PATH}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")


PROJECT_ROOT=/home/zlatko/10.semestar/Diplomski
RUN_NAME=notebook_naive_20260407T191514Z
HOURLY_DIR=/home/zlatko/10.semestar/Diplomski/data/processed/station_hourly
METADATA_PATH=/home/zlatko/10.semestar/Diplomski/data/processed/experiments/artifacts/experiment_metadata.json
OUTPUT_ROOT=/home/zlatko/10.semestar/Diplomski/data/processed/results


## Helper Functions: metadata i split priprema

U ovom bloku definiramo funkcije koje rekreiraju official sample universe iz hourly CSV datoteka.
Poanta je da notebook ne koristi nikakav novi ili alternativni workflow.

In [2]:
def build_config_from_metadata(metadata: dict[str, Any]) -> ExperimentConfig:
    split_ratios = metadata.get("split_ratios", {})
    return ExperimentConfig(
        target_column=str(metadata["target_column"]),
        window_size=int(metadata["window_size"]),
        horizon=int(metadata["horizon"]),
        train_ratio=float(split_ratios["train"]),
        val_ratio=float(split_ratios["val"]),
        test_ratio=float(split_ratios["test"]),
    )


def align_and_clean_split(
    split_df: pd.DataFrame,
    feature_columns: list[str],
    config: ExperimentConfig,
) -> pd.DataFrame:
    aligned = split_df.copy()
    for column in feature_columns:
        if column not in aligned.columns:
            aligned[column] = np.nan
    cleaned, _ = clean_split_for_windows(
        split_df=aligned,
        feature_columns=feature_columns,
        target_column=config.target_column,
        time_column=config.time_column,
    )
    return cleaned


def window_sample_count(dataframe: pd.DataFrame, window_size: int, horizon: int) -> int:
    return max(len(dataframe) - window_size - horizon + 1, 0)


def build_clean_splits_for_valid_stations(
    station_splits: dict[str, SplitDataFrame],
    valid_station_ids: list[str],
    feature_columns: list[str],
    config: ExperimentConfig,
) -> dict[str, SplitDataFrame]:
    cleaned_splits: dict[str, SplitDataFrame] = {}
    for station_id in valid_station_ids:
        split_frames = station_splits[station_id]
        cleaned_splits[station_id] = SplitDataFrame(
            train=align_and_clean_split(split_frames.train, feature_columns, config),
            val=align_and_clean_split(split_frames.val, feature_columns, config),
            test=align_and_clean_split(split_frames.test, feature_columns, config),
        )
    return cleaned_splits


def prepare_naive_inputs(hourly_dir: Path, metadata_path: Path) -> dict[str, Any]:
    metadata = load_experiment_metadata(metadata_path)
    validate_official_task(metadata, expected_feature_count=SUPPORTED_FEATURE_COUNT)
    config = build_config_from_metadata(metadata)

    bootstrap_logger = configure_run_logger(
        name="diplomski.notebooks.naive.bootstrap",
        log_path=None,
    )
    hourly_files = discover_hourly_files(hourly_dir)
    station_splits, feature_columns, _ = build_station_splits_from_hourly(
        hourly_files=hourly_files,
        config=config,
        logger=bootstrap_logger,
    )
    windows_by_station, _ = build_windows_by_station(
        station_splits=station_splits,
        feature_columns=feature_columns,
        config=config,
        logger=bootstrap_logger,
    )

    valid_station_ids = [
        station_id
        for station_id, payload in windows_by_station.items()
        if payload.X_train.shape[0] > 0
    ]
    expected_valid_station_count = int(metadata.get("valid_station_count", -1))
    if len(valid_station_ids) != expected_valid_station_count:
        raise RuntimeError(
            "Valid station count does not match experiment_metadata.json: "
            f"expected={expected_valid_station_count}, actual={len(valid_station_ids)}"
        )

    cleaned_splits = build_clean_splits_for_valid_stations(
        station_splits=station_splits,
        valid_station_ids=valid_station_ids,
        feature_columns=feature_columns,
        config=config,
    )
    return {
        "metadata": metadata,
        "config": config,
        "station_splits": station_splits,
        "feature_columns": feature_columns,
        "valid_station_ids": valid_station_ids,
        "cleaned_splits": cleaned_splits,
        "hourly_files": hourly_files,
    }


## Helper Functions: baseline predikcije i output

Ove funkcije pretvaraju ociscene splitove u predikcije, metrike i artefakte.
Naive logika je namjerno jednostavna kako bi se jasno vidjelo odakle dolazi svaka predikcija.

In [3]:
def build_predictions_for_split(
    cleaned_split: pd.DataFrame,
    station_id: str,
    method_name: str,
    config: ExperimentConfig,
) -> pd.DataFrame:
    sample_count = window_sample_count(cleaned_split, config.window_size, config.horizon)
    if sample_count == 0:
        return build_prediction_frame(
            t=np.asarray([], dtype=object),
            station_ids=np.asarray([], dtype=object),
            y_true=np.asarray([], dtype=float),
            y_pred=np.asarray([], dtype=float),
            method_name=method_name,
        )

    target_values = cleaned_split.loc[:, config.target_column].to_numpy(dtype=np.float32)
    time_values = cleaned_split.loc[:, config.time_column].to_numpy()

    y_true_values: list[float] = []
    y_pred_values: list[float] = []
    target_times: list[Any] = []
    for start_idx in range(sample_count):
        end_idx = start_idx + config.window_size
        target_idx = end_idx + config.horizon - 1
        y_true = float(target_values[target_idx])

        if method_name == "naive_last_value":
            y_pred = float(target_values[end_idx - 1])
        elif method_name == "naive_seasonal_24h":
            y_pred = float(target_values[start_idx])
        else:
            raise ValueError(f"Unsupported baseline '{method_name}'.")

        y_true_values.append(y_true)
        y_pred_values.append(y_pred)
        target_times.append(time_values[target_idx])

    station_ids = np.full((sample_count,), station_id, dtype=object)
    return build_prediction_frame(
        t=np.asarray(target_times, dtype=object),
        station_ids=station_ids,
        y_true=np.asarray(y_true_values, dtype=float),
        y_pred=np.asarray(y_pred_values, dtype=float),
        method_name=method_name,
    )


def build_predictions_by_split(
    cleaned_splits: dict[str, SplitDataFrame],
    method_name: str,
    config: ExperimentConfig,
) -> dict[str, pd.DataFrame]:
    by_split: dict[str, list[pd.DataFrame]] = {"train": [], "val": [], "test": []}
    for station_id, split_frames in cleaned_splits.items():
        by_split["train"].append(build_predictions_for_split(split_frames.train, station_id, method_name, config))
        by_split["val"].append(build_predictions_for_split(split_frames.val, station_id, method_name, config))
        by_split["test"].append(build_predictions_for_split(split_frames.test, station_id, method_name, config))

    return {
        split_name: pd.concat(frames, ignore_index=True)
        for split_name, frames in by_split.items()
    }


def validate_sample_universe(
    metadata: dict[str, Any],
    feature_columns: list[str],
    valid_station_ids: list[str],
    predictions_by_split: dict[str, pd.DataFrame],
) -> None:
    metadata_feature_columns = metadata.get("feature_columns", [])
    metadata_station_ids = metadata.get("valid_station_ids", [])
    metadata_counts = metadata.get("centralized_counts", {})

    if feature_columns != metadata_feature_columns:
        raise RuntimeError("Feature column set does not match experiment_metadata.json.")
    if valid_station_ids != metadata_station_ids:
        raise RuntimeError("Valid station IDs do not match experiment_metadata.json.")

    actual_counts = {split_name: int(frame.shape[0]) for split_name, frame in predictions_by_split.items()}
    expected_counts = {split_name: int(metadata_counts[split_name]) for split_name in ("train", "val", "test")}
    if actual_counts != expected_counts:
        raise RuntimeError(
            "Naive baseline sample counts do not match official centralized counts: "
            f"expected={expected_counts}, actual={actual_counts}"
        )


def ensure_finite_predictions(predictions_by_split: dict[str, pd.DataFrame]) -> None:
    numeric_columns = ("y_true", "y_pred", "error", "abs_error")
    for split_name, frame in predictions_by_split.items():
        if frame.empty:
            raise RuntimeError(f"Prediction frame for split '{split_name}' is empty.")
        for column in numeric_columns:
            values = frame.loc[:, column].to_numpy(dtype=float)
            if not np.isfinite(values).all():
                raise RuntimeError(
                    f"Non-finite values detected in split '{split_name}' column '{column}'."
                )


def run_baseline_method(method_name: str, prepared: dict[str, Any]) -> dict[str, Any]:
    predictions_by_split = build_predictions_by_split(
        cleaned_splits=prepared["cleaned_splits"],
        method_name=method_name,
        config=prepared["config"],
    )
    validate_sample_universe(
        metadata=prepared["metadata"],
        feature_columns=prepared["feature_columns"],
        valid_station_ids=prepared["valid_station_ids"],
        predictions_by_split=predictions_by_split,
    )
    ensure_finite_predictions(predictions_by_split)

    split_counts = {split_name: int(frame.shape[0]) for split_name, frame in predictions_by_split.items()}
    metrics_by_split = {
        split_name: compute_regression_metrics(
            frame.loc[:, "y_true"].to_numpy(dtype=float),
            frame.loc[:, "y_pred"].to_numpy(dtype=float),
        )
        for split_name, frame in predictions_by_split.items()
    }
    return {
        "method_name": method_name,
        "predictions_by_split": predictions_by_split,
        "split_counts": split_counts,
        "metrics_by_split": metrics_by_split,
    }


def save_baseline_artifacts(
    method_name: str,
    run_name: str,
    generated_at_utc: str,
    hourly_dir: Path,
    metadata_path: Path,
    output_root: Path,
    prepared: dict[str, Any],
    run_result: dict[str, Any],
) -> Path:
    run_dir = output_root / "centralized" / method_name / run_name
    logger = configure_run_logger(
        name=f"diplomski.notebooks.naive.{method_name}.{run_name}",
        log_path=run_dir / "run.log",
    )
    logger.info("Saving notebook baseline run: method=%s run_name=%s", method_name, run_name)

    results_payload = build_results_payload(
        run_name=run_name,
        stage="baseline",
        scenario="centralized",
        method=method_name,
        generated_at_utc=generated_at_utc,
        epochs_completed=0,
        best_epoch=None,
        target_column=prepared["config"].target_column,
        window_size=prepared["config"].window_size,
        horizon=prepared["config"].horizon,
        experiment_metadata_path=metadata_path,
        valid_station_ids=prepared["valid_station_ids"],
        split_counts=run_result["split_counts"],
        metrics_by_split=run_result["metrics_by_split"],
        training_info={
            "trained": False,
            "early_stopped": False,
            "best_val_rmse": float(run_result["metrics_by_split"]["val"]["rmse"]),
            "parameter_count": 0,
        },
        communication_info={
            "communication_enabled": False,
            "logical_client_count": len(prepared["valid_station_ids"]),
            "round_count": 0,
            "messages_up": 0,
            "messages_down": 0,
            "payload_bytes_up": 0,
            "payload_bytes_down": 0,
            "payload_bytes_total": 0,
            "model_state_bytes": 0,
        },
        dataset_dir=None,
        hourly_dir=hourly_dir,
    )

    save_results_payload(results_payload, run_dir / "results.json")
    save_predictions_csv(run_result["predictions_by_split"]["test"], run_dir / "test_predictions.csv")
    write_empty_communication_rounds(run_dir / "communication_rounds.csv")

    logger.info(
        "Saved outputs to %s | train=%d val=%d test=%d | test_rmse=%.6f",
        run_dir,
        run_result["split_counts"]["train"],
        run_result["split_counts"]["val"],
        run_result["split_counts"]["test"],
        run_result["metrics_by_split"]["test"]["rmse"],
    )
    return run_dir


## 1. Ucitavanje metadata i validacija taska

Prvo potvrdujemo da notebook radi bas nad sluzbenim eksperimentalnim setupom.
To smanjuje rizik da tijekom demonstracije ili kasnijeg rada nehotice koristis drugi task ili drugi feature set.

In [4]:
prepared = prepare_naive_inputs(HOURLY_DIR, METADATA_PATH)
metadata = prepared["metadata"]
config = prepared["config"]

task_overview = pd.DataFrame(
    [
        {"field": "window_size", "value": config.window_size},
        {"field": "horizon", "value": config.horizon},
        {"field": "target_column", "value": config.target_column},
        {"field": "feature_count", "value": len(prepared["feature_columns"])},
        {"field": "hourly_files", "value": len(prepared["hourly_files"])},
        {"field": "valid_station_count", "value": len(prepared["valid_station_ids"])},
    ]
)
task_overview


2026-04-07 21:15:16,673 | WARNING | 1001 dropped rows during split cleaning: train=0 val=0 test=881
2026-04-07 21:15:16,730 | WARNING | 1002 dropped rows during split cleaning: train=0 val=0 test=881
2026-04-07 21:15:16,784 | WARNING | 1003 dropped rows during split cleaning: train=241 val=0 test=881
2026-04-07 21:15:16,845 | WARNING | 1004 dropped rows during split cleaning: train=635 val=64 test=320
2026-04-07 21:15:16,890 | WARNING | 1005 dropped rows during split cleaning: train=3040 val=217 test=993
2026-04-07 21:15:16,956 | WARNING | 1006 dropped rows during split cleaning: train=9 val=0 test=0
2026-04-07 21:15:17,005 | WARNING | 1007 dropped rows during split cleaning: train=3040 val=217 test=1067
2026-04-07 21:15:17,074 | WARNING | 1008 dropped rows during split cleaning: train=0 val=0 test=881
2026-04-07 21:15:17,150 | WARNING | 1009 dropped rows during split cleaning: train=81 val=0 test=0
2026-04-07 21:15:17,195 | WARNING | 1010 dropped rows during split cleaning: train=3040

,field,value
0,window_size,24
1,horizon,1
2,target_column,volume
3,feature_count,19
4,hourly_files,51
5,valid_station_count,40


## 2. Pregled stanica i splitova

Ovdje gledamo kako izgledaju train/val/test splitovi prije i nakon ciscenja.
To je koristan korak jer odmah pokazuje zasto neke stanice mogu imati manje valjanih uzoraka od drugih.

In [5]:
station_overview_rows = []
for station_id in prepared["valid_station_ids"][:10]:
    raw_split = prepared["station_splits"][station_id]
    clean_split = prepared["cleaned_splits"][station_id]
    station_overview_rows.append(
        {
            "station_id": station_id,
            "train_raw": len(raw_split.train),
            "train_clean": len(clean_split.train),
            "val_raw": len(raw_split.val),
            "val_clean": len(clean_split.val),
            "test_raw": len(raw_split.test),
            "test_clean": len(clean_split.test),
        }
    )

station_overview = pd.DataFrame(station_overview_rows)
station_overview


,station_id,train_raw,train_clean,val_raw,val_clean,test_raw,test_clean
0,1001,3040,3040,217,217,1087,206
1,1002,3040,3040,217,217,1087,206
2,1003,3040,2799,217,217,1087,206
3,1004,3040,2405,217,153,1087,767
4,1006,3040,3031,217,217,1087,1087
5,1008,3040,3040,217,217,1087,206
6,1009,3040,2959,217,217,1087,1087
7,1011,3040,2959,217,217,1087,1087
8,1013,3040,2798,217,217,1087,1087
9,1014,3040,2960,217,217,1087,1087


## 3. Kako nastaje jedan forecasting sample

Za `24h -> 1h ahead` uzimamo 24 uzastopna hourly retka kao povijest, a target je sljedeci sat.
U ovom primjeru gledamo prvi train window prve validne stanice.

In [6]:
example_station_id = prepared["valid_station_ids"][0]
example_split = prepared["cleaned_splits"][example_station_id].train.reset_index(drop=True)
example_start = 0
example_end = example_start + config.window_size
example_target_idx = example_end + config.horizon - 1

example_frame = example_split.loc[
    example_start:example_target_idx,
    [config.time_column, config.target_column],
].copy()
example_frame["role"] = ["history"] * config.window_size + ["target"]
example_frame


,time,volume,role
0,2022-09-01 00:00:00,4.083333,history
1,2022-09-01 01:00:00,2.916667,history
2,2022-09-01 02:00:00,5.250000,history
3,2022-09-01 03:00:00,5.250000,history
4,2022-09-01 04:00:00,4.083333,history
5,2022-09-01 05:00:00,2.916667,history
6,2022-09-01 06:00:00,5.250000,history
7,2022-09-01 07:00:00,2.916667,history
8,2022-09-01 08:00:00,7.000000,history
9,2022-09-01 09:00:00,6.416667,history


### Objasnjenje naive predikcija

- `naive_last_value` uzima zadnju poznatu vrijednost iz history prozora.
- `naive_seasonal_24h` uzima vrijednost iz istog sata prethodnog dana, sto je u ovom tasku prvi element prozora.

In [7]:
example_last_value = float(example_split.loc[example_end - 1, config.target_column])
example_seasonal = float(example_split.loc[example_start, config.target_column])
example_target = float(example_split.loc[example_target_idx, config.target_column])

pd.DataFrame(
    [
        {"item": "station_id", "value": example_station_id},
        {"item": "naive_last_value_prediction", "value": example_last_value},
        {"item": "naive_seasonal_24h_prediction", "value": example_seasonal},
        {"item": "true_target", "value": example_target},
    ]
)


,item,value
0,station_id,1001
1,naive_last_value_prediction,5.25
2,naive_seasonal_24h_prediction,4.083333
3,true_target,4.083333


## 4. Izracun `naive_last_value`

Sada izgradujemo predikcije za train, val i test nad svim validnim stanicama i racunamo metrike.

In [8]:
last_value_run = run_baseline_method("naive_last_value", prepared)
pd.DataFrame(
    [
        {"split": split_name, **metrics}
        for split_name, metrics in last_value_run["metrics_by_split"].items()
    ]
)


,split,rmse,mae,r2,sample_count
0,train,408.644653,116.536681,0.879040,110642
1,val,351.096900,82.271780,0.865247,6824
2,test,337.208651,85.323655,0.877898,30275


## 5. Izracun `naive_seasonal_24h`

Ovaj baseline koristi dnevnu sezonalnost i obicno je jaci kada postoji slican obrazac izmedu istih sati razlicitih dana.

In [9]:
seasonal_run = run_baseline_method("naive_seasonal_24h", prepared)
pd.DataFrame(
    [
        {"split": split_name, **metrics}
        for split_name, metrics in seasonal_run["metrics_by_split"].items()
    ]
)


,split,rmse,mae,r2,sample_count
0,train,555.347789,175.222474,0.776601,110642
1,val,299.568025,94.335189,0.901898,6824
2,test,275.568596,95.596958,0.918458,30275


## 6. Usporedba baselinea

Spajamo metrike u jednu tablicu kako bi se lakse vidjelo koji baseline je jaci na kojem splitu.

In [10]:
baseline_runs = {
    "naive_last_value": last_value_run,
    "naive_seasonal_24h": seasonal_run,
}

summary_rows = []
for method_name, run_result in baseline_runs.items():
    for split_name, metrics in run_result["metrics_by_split"].items():
        summary_rows.append(
            {
                "method_name": method_name,
                "split": split_name,
                "rmse": metrics["rmse"],
                "mae": metrics["mae"],
                "r2": metrics["r2"],
                "sample_count": metrics["sample_count"],
            }
        )

metrics_summary = pd.DataFrame(summary_rows)
metrics_summary


,method_name,split,rmse,mae,r2,sample_count
0,naive_last_value,train,408.644653,116.536681,0.879040,110642
1,naive_last_value,val,351.096900,82.271780,0.865247,6824
2,naive_last_value,test,337.208651,85.323655,0.877898,30275
3,naive_seasonal_24h,train,555.347789,175.222474,0.776601,110642
4,naive_seasonal_24h,val,299.568025,94.335189,0.901898,6824
5,naive_seasonal_24h,test,275.568596,95.596958,0.918458,30275


### Kratki pogled u test predikcije

Ovdje se vidi canonical prediction schema koji kasnije mogu koristiti i drugi modeli.

In [11]:
test_preview = pd.concat(
    [
        last_value_run["predictions_by_split"]["test"].head(5),
        seasonal_run["predictions_by_split"]["test"].head(5),
    ],
    ignore_index=True,
)
test_preview


,t,station_id,y_true,y_pred,error,abs_error,method_name
0,2023-01-15 17:00:00,1001,0.0,0.00,0.00,0.00,naive_last_value
1,2023-01-15 18:00:00,1001,0.0,0.00,0.00,0.00,naive_last_value
2,2023-01-15 19:00:00,1001,0.0,0.00,0.00,0.00,naive_last_value
3,2023-01-15 20:00:00,1001,0.0,0.00,0.00,0.00,naive_last_value
4,2023-01-15 21:00:00,1001,0.0,0.00,0.00,0.00,naive_last_value
5,2023-01-15 17:00:00,1001,0.0,5.25,5.25,5.25,naive_seasonal_24h
6,2023-01-15 18:00:00,1001,0.0,1.75,1.75,1.75,naive_seasonal_24h
7,2023-01-15 19:00:00,1001,0.0,0.00,0.00,0.00,naive_seasonal_24h
8,2023-01-15 20:00:00,1001,0.0,0.00,0.00,0.00,naive_seasonal_24h
9,2023-01-15 21:00:00,1001,0.0,0.00,0.00,0.00,naive_seasonal_24h


## 7. Spremanje artefakata

Za svaki baseline spremamo iste glavne outpute:
- `results.json`
- `test_predictions.csv`
- `communication_rounds.csv`
- `run.log`

In [12]:
saved_runs: dict[str, Path] = {}
for method_name, run_result in baseline_runs.items():
    saved_runs[method_name] = save_baseline_artifacts(
        method_name=method_name,
        run_name=RUN_NAME,
        generated_at_utc=GENERATED_AT_UTC,
        hourly_dir=HOURLY_DIR,
        metadata_path=METADATA_PATH,
        output_root=OUTPUT_ROOT,
        prepared=prepared,
        run_result=run_result,
    )

pd.DataFrame(
    [{"method_name": key, "run_dir": str(value)} for key, value in saved_runs.items()]
)


2026-04-07 21:15:21,992 | INFO | Saving notebook baseline run: method=naive_last_value run_name=notebook_naive_20260407T191514Z
2026-04-07 21:15:22,402 | INFO | Saved outputs to /home/zlatko/10.semestar/Diplomski/data/processed/results/centralized/naive_last_value/notebook_naive_20260407T191514Z | train=110642 val=6824 test=30275 | test_rmse=337.208651
2026-04-07 21:15:22,405 | INFO | Saving notebook baseline run: method=naive_seasonal_24h run_name=notebook_naive_20260407T191514Z
2026-04-07 21:15:22,680 | INFO | Saved outputs to /home/zlatko/10.semestar/Diplomski/data/processed/results/centralized/naive_seasonal_24h/notebook_naive_20260407T191514Z | train=110642 val=6824 test=30275 | test_rmse=275.568596


,method_name,run_dir
0,naive_last_value,/home/zlatko/10.semestar/Diplomski/data/proces...
1,naive_seasonal_24h,/home/zlatko/10.semestar/Diplomski/data/proces...


### Popis spremljenih datoteka

Na kraju mozemo brzo provjeriti sto je točno zapisano na disk.

In [13]:
artifact_rows = []
for method_name, run_dir in saved_runs.items():
    for artifact_path in sorted(run_dir.iterdir()):
        artifact_rows.append(
            {
                "method_name": method_name,
                "artifact": artifact_path.name,
                "path": str(artifact_path),
            }
        )

artifact_index = pd.DataFrame(artifact_rows)
artifact_index


,method_name,artifact,path
0,naive_last_value,communication_rounds.csv,/home/zlatko/10.semestar/Diplomski/data/proces...
1,naive_last_value,results.json,/home/zlatko/10.semestar/Diplomski/data/proces...
2,naive_last_value,run.log,/home/zlatko/10.semestar/Diplomski/data/proces...
3,naive_last_value,test_predictions.csv,/home/zlatko/10.semestar/Diplomski/data/proces...
4,naive_seasonal_24h,communication_rounds.csv,/home/zlatko/10.semestar/Diplomski/data/proces...
5,naive_seasonal_24h,results.json,/home/zlatko/10.semestar/Diplomski/data/proces...
6,naive_seasonal_24h,run.log,/home/zlatko/10.semestar/Diplomski/data/proces...
7,naive_seasonal_24h,test_predictions.csv,/home/zlatko/10.semestar/Diplomski/data/proces...
